# 01 · BioEMU Ensemble Generation & Pocket Detection

**Pipeline stage:** Conformational sampling → pocket identification → persistent/cryptic labelling

This notebook covers:
1. Generating equilibrium conformational ensembles with **BioEMU**
2. Detecting candidate pockets per frame via `EnsemblePocketFinder`
3. Classifying each pocket as **persistent** or **cryptic**

> **Runtime:** ~15 min on a T4 GPU for 200 samples per protein.  
> Set `QUICK_RUN = True` to use 50 samples for fast iteration.


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bioemu.sample import main as bioemu_sample
from bioemu_pocket.pocket_finder import EnsemblePocketFinder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
SEED = 16
np.random.seed(SEED)

## 1 · Protein Selection

We benchmark on five small, well-studied proteins:

| Name | Length | Notes |
|---|---|---|
| chignolin | 10 aa | β-hairpin miniprotein |
| villin HP35 | 35 aa | Three-helix bundle |
| protein G (β1) | 56 aa | Mixed α/β fold |
| trp-cage | 20 aa | Fastest-folding miniprotein |
| WW domain | 18 aa | β-sheet signalling module |

BioEMU generates each ensemble 10,000× faster than conventional MD while preserving thermodynamic accuracy (Lin et al., 2024).


In [ ]:
PROTEINS = {
    "chignolin": "GYDPETGTWG",
    "villin": "MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF",
    "protein_g": "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "trp_cage": "NLYIQWLKDGGPSSGRPPPS",
    "ww_domain": "IPGWEWECEDQIHWWEDG",
}

QUICK_RUN = False  # True -> 50 samples (fast smoke-test)
NUM_SAMPLES = 50 if QUICK_RUN else 200
ROOT_OUT = Path("./bioemu_runs")

for name, seq in PROTEINS.items():
    outdir = ROOT_OUT / f"{name}_n{NUM_SAMPLES}_s{SEED}"
    outdir.mkdir(parents=True, exist_ok=True)
    print(f"\n── {name} ({len(seq)} aa) ──")
    bioemu_sample(
        sequence=seq, num_samples=NUM_SAMPLES, output_dir=str(outdir), seed=SEED
    )
    print(f"   -> {outdir}")

## 2 · Ensemble Pocket Detection

`EnsemblePocketFinder` iterates every BioEMU frame and:

1. Computes Cα pairwise distances.
2. Selects candidate residues by neighbour count and burial depth (size-adaptive).
3. Clusters candidates via **Ward linkage** (proteins > 20 aa) or **DBSCAN** (≤ 20 aa).
4. Annotates each cluster with hydrophobicity, aromaticity, depth, and estimated volume.

### Cryptic vs. persistent classification

A pocket is *persistent* when its spatial centroid appears in ≥ 30 % of frames:

$$\text{is\_cryptic} = \mathbb{1}\!\left[\frac{N_{\text{frames with pocket at bin}}}{N_{\text{frames}}} < 0.30\right]$$


In [ ]:
results_summary = []
ALL_POCKETS = {}

for run_dir in sorted(ROOT_OUT.glob("*_n*_s*")):
    name = run_dir.name.split("_n")[0]
    top, xtc = run_dir / "topology.pdb", run_dir / "samples.xtc"
    if not (top.exists() and xtc.exists()):
        print(f"[SKIP] {name}: missing topology/xtc")
        continue

    print(f"\n─── {name} ───")
    finder = EnsemblePocketFinder(
        top_path=str(top), xtc_path=str(xtc), min_residues=3, distance_cutoff=6.0
    )
    pocket_data, pockets_per_conf = finder.run(max_frames=NUM_SAMPLES)
    finder.mark_cryptic(pocket_data, n_frames_used=min(NUM_SAMPLES, finder.n_frames))

    ALL_POCKETS[name] = pocket_data
    results_summary.append(
        {
            "protein": name,
            "frames": finder.n_frames,
            "total_pockets": len(pocket_data),
            "cryptic": sum(p["is_cryptic"] for p in pocket_data),
            "avg_pockets_per_frame": round(np.mean(pockets_per_conf), 2),
        }
    )

df_summary = pd.DataFrame(results_summary)
print("\n", df_summary.to_string(index=False))

## 3 · Visualisation

In [ ]:
fig, axes = plt.subplots(
    len(results_summary), 3, figsize=(14, 4 * len(results_summary))
)
if len(results_summary) == 1:
    axes = axes[None, :]

for row, info in enumerate(results_summary):
    name = info["protein"]
    pocket_data = ALL_POCKETS[name]
    ppc = [
        sum(1 for p in pocket_data if p["conf_id"] == i) for i in range(info["frames"])
    ]
    sizes = [p["size"] for p in pocket_data]
    n_p = sum(1 for p in pocket_data if not p["is_cryptic"])
    n_c = len(pocket_data) - n_p

    axes[row, 0].hist(ppc, bins=max(5, len(set(ppc))), alpha=0.8, edgecolor="black")
    axes[row, 0].axvline(
        np.mean(ppc), color="red", ls="--", lw=1, label=f"mu={np.mean(ppc):.1f}"
    )
    axes[row, 0].set_title(f"{name}: Pockets / Conformation")
    axes[row, 0].legend()
    axes[row, 1].hist(
        sizes, bins=min(12, max(5, len(set(sizes)))), alpha=0.8, edgecolor="black"
    )
    axes[row, 1].set_title("Pocket Size (residues)")
    axes[row, 2].bar(
        ["Persistent", "Cryptic"],
        [n_p, n_c],
        color=["#2ecc71", "#e74c3c"],
        edgecolor="black",
        alpha=0.85,
    )
    axes[row, 2].set_title("Classification")

plt.suptitle("BioEMU: Pocket Detection Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4 · Save for Next Notebook

In [ ]:
import pickle

out = Path("./outputs")
out.mkdir(exist_ok=True)
with open(out / "all_pockets.pkl", "wb") as f:
    pickle.dump(ALL_POCKETS, f)
df_summary.to_csv(out / "pocket_summary.csv", index=False)
print("Saved outputs/all_pockets.pkl and pocket_summary.csv")